# 02 – Wie beeinflusst ein Prompt die Antwort?

**Lernziel:** Wir untersuchen, wie sich **Auftrag, Kontext, Anforderungen und Rolle** auf die Ausgabe eines LLM auswirken.

Dafür verwenden wir **dasselbe Modell und dieselbe Grundaufgabe**. Nur die Eingabe verändert sich. So wird sichtbar, welchen Unterschied die Formulierung macht.

**Aufgabe für alle Versuche:** Unterstützung beim Erstellen eines Python-REST-Clients.


## Init – Ein Modell, drei Arten zu fragen

Wir nutzen einen lokalen Ollama-Server und das Modell `gemma3:4b`. Die Verbindungsdetails spielen für die folgende Untersuchung keine Rolle. Der folgende Block ist die gemeinsame Vorbereitung.


In [ ]:
from openai import OpenAI
from IPython.display import Markdown, display

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama", timeout=180.0)
MODELL = "gemma3:4b"

def fragen(prompt):
    antwort = client.chat.completions.create(
        model=MODELL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=550,
    )
    return antwort.choices[0].message.content or "(Keine Textantwort)"


## 1. Offener Prompt – was fehlt dem Modell?

**Fragestellung:** Was entsteht, wenn wir nur ein grobes Ziel angeben?

**Beobachtet:** Welchen HTTP-Endpunkt wählt das Modell? Welche Bibliothek? Fehlen Fehlerbehandlung, Erklärungen oder Informationen zur Verwendung?


In [ ]:
AUFGABE = "Schreibe ein kurzes Python-Beispiel für einen REST-Client."

antwort_offen = fragen(AUFGABE)
display(Markdown(antwort_offen))


## 2. Präziser Prompt – Anforderungen statt Vermutungen

**Fragestellung:** Verbessert sich die Nutzbarkeit, wenn wir klar sagen, **für wen** die Lösung ist und **was sie können soll**?

Wir bleiben beim REST-Client, machen das erwartete Ergebnis aber überprüfbar.


In [ ]:
PRAEZISER_PROMPT = """Schreibe für Python-Einsteiger einen einfachen REST-Client.
- Verwende die Bibliothek requests.
- Sende einen GET-Request an https://httpbin.org/get.
- Prüfe den HTTP-Status und behandle Verbindungsfehler.
- Gib ein möglichst kurzes, vollständiges Beispiel aus.
- Erkläre anschliessend in drei Sätzen, was der Code macht.
"""

antwort_praezise = fragen(PRAEZISER_PROMPT)
display(Markdown(antwort_praezise))


## 3. Rollen-Prompt – das Modell als Lernbegleiter

**Fragestellung:** Was passiert, wenn wir nicht eine fertige Lösung verlangen, sondern **Lernunterstützung**?

Der nächste Prompt gibt dem Modell die Rolle eines Tutors. Er fordert eine feste Gliederung und verlangt ausdrücklich, die Lösung nicht vorwegzunehmen. Das folgt der Idee eines [Masterprompts für selbstständiges Lernen](https://gitlab.com/ch-tbz-it/Stud/allgemein/ki-kompetenz).


In [ ]:
TUTOR_PROMPT = """[Rolle]
Du bist mein Tutor für Programmierung. Hilf mir, selbstständig zu lernen.
Gib keine vollständige Musterlösung für die Aufgabe.

[Auftrag]
Ich möchte lernen, einen einfachen Python-REST-Client zu erstellen.

[Antwortstruktur]
1. Grundlagen: Welche Begriffe muss ich kennen?
2. Konzeptverständnis: Wie funktioniert ein REST-Request grundsätzlich?
3. Nächster Schritt: Welche kleine Aufgabe kann ich selbst lösen?
4. Lernportfolio: Was soll ich zu meinem Lernfortschritt festhalten?

Schreibe verständlich, konkret und knapp.
"""

antwort_tutor = fragen(TUTOR_PROMPT)
display(Markdown(antwort_tutor))


## Auswertung – was hat sich verändert?

| Prompt | Was haben wir vorgegeben? | Worauf achten wir? |
|---|---|---|
| Offen | Nur die Aufgabe | Annahmen, Lücken, Variationen |
| Präzise | Zielgruppe, Anforderungen, Format | Passung zum Auftrag, Vollständigkeit |
| Tutor | Rolle, Lernziel, Antwortstruktur | Hilfe zum Verstehen statt fertigem Code |

**Erkenntnis:** Ein Prompt ist nicht bloss eine Frage. Er steuert, welche Aufgabe das Modell lösen, welche Grenzen es beachten und wie es antworten soll. Mehr Text allein garantiert keine bessere Antwort.

**Eigener Versuch:** Verändert beim präzisen Prompt nur **eine** Anforderung (z. B. Zielgruppe oder Ausgabeformat) und führt ihn erneut aus. Beschreibt anschliessend, was tatsächlich anders wurde.

**Optional:** Ändert im Initialisierungsblock `temperature` oder `max_tokens`. Beobachtet, ob sich Varianz beziehungsweise Ausgabelänge verändert. Antwortzeitvergleiche sind nur bedingt aussagekräftig, da das erste Laden eines lokalen Modells länger dauern kann.
